# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Same Refresh/Decline feature set as `w03_feature_leakage_check`: `imp_prev30`, `pos_prev30`, `ctr_prev30`, `visible_queries`, `rare_share`, `anon_share`, `top_query_share`. Rebuilt here from scratch since each notebook is meant to stand on its own (per the course setup convention).

**What to expect before looking:** impressions and clicks are almost always heavy-tailed in search data — a small number of pages carry a large share of total traffic. Plotting these on a linear axis usually just shows one tall bar and a flat line, so impressions/clicks are viewed on a log axis below. Position and CTR are bounded and usually closer to well-behaved.


In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn matplotlib

import os, getpass
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           ANY_VALUE(rare_impressions_share)        AS rare_share,
           ANY_VALUE(anonymized_impressions_share)  AS anon_share,
           MAX(impressions_90d)                     AS top_query_impressions,
           SUM(impressions_90d)                     AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features.merge(qsignals, on='content_hash_id', how='left')
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
audit = data.dropna(subset=FEATURE_COLS + ['is_declining']).copy()
print(f'{len(audit):,} rows ready for audit')

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
log_cols  = ['imp_prev30', 'clk_prev30']
lin_cols  = ['pos_prev30', 'ctr_prev30', 'visible_queries', 'rare_share']
for ax, col in zip(axes.flat[:2], log_cols):
    ax.hist(audit[col].clip(lower=1), bins=50)
    ax.set_xscale('log'); ax.set_title(f'{col} (log x)')
for ax, col in zip(axes.flat[2:], lin_cols):
    ax.hist(audit[col], bins=50)
    ax.set_title(col)
plt.tight_layout(); plt.show()

print(audit[FEATURE_COLS].describe().T[['mean', '50%', 'std', 'max']])


## 2. Signal test #1 / #2 / #3 (verdict each)

Median-split test on each signal: does the “high” half of the group show a meaningfully different decline rate than the “low” half? Verdict labels: **CONFIRMED** (matches the intuitive hypothesis), **OPPOSITE** (reverses it), **MIXED** (direction holds but the gap is small/noisy), **FALSE** (no real difference).

- **Signal 1 — `visible_queries`:** hypothesis — pages ranking for *more* distinct queries are more resilient (more ways to keep earning impressions), so *low* `visible_queries` should see *more* decline.
- **Signal 2 — `top_query_share` (concentration):** hypothesis — pages that depend heavily on one query are more fragile, so *high* concentration should see *more* decline.
- **Signal 3 — `ctr_prev30`:** hypothesis — pages already under-clicking relative to peers are weaker performers overall, so *low* `ctr_prev30` should see *more* decline.


In [ ]:
def median_split_test(df, col, label='is_declining', higher_means='more decline'):
    med = df[col].median()
    low_rate  = df.loc[df[col] <  med, label].mean()
    high_rate = df.loc[df[col] >= med, label].mean()
    gap = high_rate - low_rate
    print(f'{col:<18} low-half decline rate: {low_rate:.3f}   high-half: {high_rate:.3f}   gap: {gap:+.3f}   (n={len(df):,})')
    return gap

print('Signal 1 -- visible_queries (expect: LOW visible_queries -> MORE decline, i.e. negative gap)')
gap1 = median_split_test(audit, 'visible_queries')
verdict1 = 'CONFIRMED' if gap1 < -0.02 else ('OPPOSITE' if gap1 > 0.02 else 'MIXED')
print(f'  VERDICT: {verdict1}\n')

print('Signal 2 -- top_query_share / concentration (expect: HIGH concentration -> MORE decline, positive gap)')
gap2 = median_split_test(audit, 'top_query_share')
verdict2 = 'CONFIRMED' if gap2 > 0.02 else ('OPPOSITE' if gap2 < -0.02 else 'MIXED')
print(f'  VERDICT: {verdict2}\n')

print('Signal 3 -- ctr_prev30 (expect: LOW ctr_prev30 -> MORE decline, negative gap)')
gap3 = median_split_test(audit, 'ctr_prev30')
verdict3 = 'CONFIRMED' if gap3 < -0.02 else ('OPPOSITE' if gap3 > 0.02 else 'MIXED')
print(f'  VERDICT: {verdict3}')

# NOTE: the +/-0.02 threshold is an arbitrary but stated cutoff for "meaningfully different" --
# swap in a proper significance test (e.g. a two-proportion z-test) if you want a stricter bar.


## 3. The flag-linked test

**Assumption stated up front:** I don't have access to FlyRank's actual internal flag definitions, so I'm testing the most common real-world refresh-flag heuristic instead of guessing at proprietary logic: *“a page whose average search position is getting worse is treated as at-risk.”* If your `w04_baseline_score` used a different reason code, swap the column/threshold below and re-run — the test pattern stays the same.

**Test:** split pages into “worse position” (bottom half by rank, i.e. higher `pos_prev30` number = worse) vs “better position”, and compare decline rates — same median-split pattern as Section 2, applied to the specific signal a real flag would key off.


In [ ]:
print('Flag-linked signal -- pos_prev30 (expect: WORSE position, i.e. higher number, -> MORE decline)')
gap_flag = median_split_test(audit, 'pos_prev30')
verdict_flag = 'CONFIRMED' if gap_flag > 0.02 else ('OPPOSITE' if gap_flag < -0.02 else 'MIXED')
print(f'  VERDICT: {verdict_flag}')

# Break it into quartiles too -- a single median split can hide a threshold effect
audit['pos_quartile'] = pd.qcut(audit['pos_prev30'], 4, labels=['Q1 best', 'Q2', 'Q3', 'Q4 worst'])
print()
print(audit.groupby('pos_quartile', observed=True)['is_declining'].agg(['mean', 'count']))


## 4. What this means in practice

*(Fill this in with your own printed verdicts once you've run Sections 2–3 — the sentence templates below just show the shape of a good answer.)*

- If position-based flagging is **CONFIRMED**: a content team's existing “position slipping” alert is worth trusting as a first-pass triage signal — but it should not be the only signal, since the gap is directional, not deterministic (see Section 3's quartile breakdown for how much it actually moves).
- If query concentration is **CONFIRMED**: pages that earn almost all their traffic from one query are a reasonable early-warning group to prioritize for diversification work, ahead of any visible decline.
- Any signal that comes back **OPPOSITE** or **FALSE** should be flagged to the team as “do not use this as a stand-alone flag” — that's a real, useful finding, not a failed test.


In [ ]:
summary = pd.DataFrame({
    'signal':  ['visible_queries', 'top_query_share', 'ctr_prev30', 'pos_prev30 (flag-linked)'],
    'gap':     [gap1, gap2, gap3, gap_flag],
    'verdict': [verdict1, verdict2, verdict3, verdict_flag],
})
print(summary.to_string(index=False))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.